# Đánh giá & So sánh 3 mô hình phát hiện biển báo (Zalo AI 2020)

Notebook này chấm điểm **cùng lúc 3 mô hình** trên **cùng một tập Hold-out Test**, phục vụ bảng so sánh trong báo cáo môn học.

| Mô hình | Thư viện | `imgsz` khi suy luận | Nguồn spec |
|---|---|---|---|
| YOLOv8s-P2 | `ultralytics` | `1280` | `models_specs.md` mục 1.2 |
| Faster R-CNN ResNet50-FPN | `torchvision` (PyTorch thuần) | cạnh nhỏ nhất `800` (tự scale) | `models_specs.md` mục 2.2 |
| RT-DETR-L | `ultralytics` | `640` | `models_specs.md` mục 3.2 |

**Các chỉ số sẽ đo:**
1. `mAP@50` và `mAP@50-95` (thêm `mAP_small` vì đây là bài toán Small Object).
2. Tốc độ suy luận (ms/ảnh và FPS), **có bước warm-up GPU** để số liệu công bằng.
3. Ma trận nhầm lẫn (Confusion Matrix) cho 7 lớp biển báo.
4. Bảng tổng hợp cuối cùng dưới dạng Pandas DataFrame.

> ⚠️ **Lưu ý trung thực bắt buộc ghi vào báo cáo:** Tập Test được tái lập từ 20% dữ liệu có nhãn bằng `random.seed(42)` — đúng phép chia mà YOLOv8 và RT-DETR đã dùng lúc train. Riêng Faster R-CNN lúc train dùng `random_split` **không set seed** và cắt 90/10, nên phần lớn ảnh trong tập test này nó đã học qua rồi.
>
> Cần nói rõ: cả 3 mô hình đều chỉ đọc dữ liệu từ `traffic_train/`, **không có Data Leakage** từ tập test của cuộc thi. Đây thuần túy là chênh lệch điều kiện đánh giá — điểm của Faster R-CNN sẽ pha lẫn phần "học thuộc lòng" (Overfitting), trong khi YOLOv8 và RT-DETR bị chấm trên ảnh hoàn toàn mới. Chi tiết ở mục 0.2 của `docs/05_testing_evaluation/test_KeHoach_Kaggle.md`.

## Cell 1: Cài thư viện cần thiết

Dùng `torchmetrics` để tính mAP cho **cả 3 mô hình bằng cùng một công thức**. Nếu để `ultralytics` tự tính mAP cho YOLO/RT-DETR rồi tự viết tay công thức khác cho Faster R-CNN thì con số sẽ không so sánh được với nhau.

In [ ]:
!pip install -q ultralytics torchmetrics

## Cell 2: Khai báo cấu hình chung

Toàn bộ đường dẫn và siêu tham số gom về một chỗ để dễ chỉnh sửa (tránh sửa rải rác khắp notebook).

In [ ]:
import os
import json
import glob
import time
import random

import numpy as np
import pandas as pd
import torch
import cv2
import matplotlib.pyplot as plt

# Khoa cung 1 GPU duy nhat de so lieu FPS sach va de giai thich
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"Thiet bi dang dung: {DEVICE}")

# 7 lop bien bao, thu tu lay dung theo file dataset.yaml trong train_yolov8.ipynb
CLASS_NAMES = [
    'No entry',                 # 0 - Cam nguoc chieu
    'No parking / waiting',     # 1 - Cam dung va do
    'No turning',               # 2 - Cam re
    'Max Speed',                # 3 - Gioi han toc do
    'Other prohibition signs',  # 4 - Cam con lai
    'Warning signs',            # 5 - Nguy hiem
    'Mandatory signs',          # 6 - Hieu lenh
]
NUM_CLASSES = len(CLASS_NAMES)

# Nguong suy luan: lay thap de duong cong Precision-Recall du diem tinh mAP cho chuan
CONF_FOR_MAP = 0.001
# Nguong hien thi thuc te, dung khi ve Confusion Matrix (bam theo spec Web App)
CONF_FOR_MATRIX = 0.25
IOU_NMS = 0.6      # [E2] Giu bien bao dung canh nhau
MAX_DET = 50       # [E1, E2] Toi uu luong NMS

# So anh dung cho khau do toc do
WARMUP_IMAGES = 15   # Chay nhap cho GPU "nong may" truoc khi bam gio
SPEED_IMAGES = 100   # So anh do that su

OUTPUT_DIR = '/kaggle/working'

## Cell 3: Tự động dò tìm đường dẫn dữ liệu và trọng số

Viết hàm dò tìm thay vì gõ cứng đường dẫn, vì tên thư mục trong `/kaggle/input/` phụ thuộc vào slug dataset của từng người. Cách này cũng giúp code chạy được dù bạn upload weights theo kiểu file phẳng hay có thư mục con.

In [ ]:
def tim_file_dau_tien(mau_duong_dan, mo_ta):
    """Tim file/thu muc dau tien khop voi mau glob. Bao loi ro rang neu khong thay."""
    ket_qua = glob.glob(mau_duong_dan, recursive=True)
    if not ket_qua:
        raise FileNotFoundError(
            f"Khong tim thay {mo_ta}.\n"
            f"Mau tim kiem: {mau_duong_dan}\n"
            f"Hay kiem tra lai da Add Input dung dataset chua."
        )
    return sorted(ket_qua)[0]


def tim_weight_theo_tu_khoa(tu_khoa, duoi_file):
    """Do tim file trong so theo tu khoa trong ten file (khong phan biet hoa thuong)."""
    tat_ca = glob.glob(f'/kaggle/input/**/*{duoi_file}', recursive=True)
    for duong_dan in sorted(tat_ca):
        if tu_khoa.lower() in os.path.basename(duong_dan).lower():
            return duong_dan
    # Neu khong khop tu khoa thi thu do theo ten thu muc cha
    for duong_dan in sorted(tat_ca):
        if tu_khoa.lower() in duong_dan.lower():
            return duong_dan
    raise FileNotFoundError(
        f"Khong tim thay file {duoi_file} chua tu khoa '{tu_khoa}'.\n"
        f"Cac file {duoi_file} dang co trong /kaggle/input/: {tat_ca}"
    )


# Duong dan toi bo du lieu goc da co san tren Kaggle
JSON_PATH = tim_file_dau_tien(
    '/kaggle/input/**/train_traffic_sign_dataset.json',
    'file nhan COCO cua Zalo AI'
)
IMAGE_DIR = tim_file_dau_tien(
    '/kaggle/input/**/traffic_train/images',
    'thu muc anh train'
)

# Duong dan toi 3 bo trong so tu dataset ca nhan
YOLO_WEIGHT = tim_weight_theo_tu_khoa('yolo', '.pt')
RTDETR_WEIGHT = tim_weight_theo_tu_khoa('rtdetr', '.pt')
FRCNN_WEIGHT = tim_weight_theo_tu_khoa('rcnn', '.pth')

print(f"File JSON nhan   : {JSON_PATH}")
print(f"Thu muc anh      : {IMAGE_DIR}")
print(f"Weight YOLOv8-P2 : {YOLO_WEIGHT}")
print(f"Weight RT-DETR   : {RTDETR_WEIGHT}")
print(f"Weight FasterRCNN: {FRCNN_WEIGHT}")

## Cell 4: Dựng tập Hold-out Test

Tái lập **đúng** phép chia đã dùng lúc huấn luyện: `random.seed(42)` → shuffle → lấy 20% cuối. Nhờ seed cố định nên danh sách ảnh sinh ra ở đây giống hệt tập `val` mà YOLOv8 và RT-DETR đã dùng, không cần upload thêm file nào.

Nhãn gốc trong JSON có `category_id` chạy từ **1 đến 7**, nên phải trừ đi 1 để về đúng chỉ số **0 đến 6** mà cả 3 mô hình đang dùng.

In [ ]:
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

images_info = {img['id']: img for img in coco_data['images']}

# Gom cac bounding box lai theo tung anh
img_to_anns = {}
for ann in coco_data['annotations']:
    img_to_anns.setdefault(ann['image_id'], []).append(ann)

# Tai lap y het phep chia luc train (train_yolov8.ipynb / train_rtdetr.ipynb)
image_ids = list(images_info.keys())
random.seed(42)
random.shuffle(image_ids)
split_idx = int(len(image_ids) * 0.8)
test_ids = image_ids[split_idx:]

print(f"Tong so anh co nhan: {len(image_ids)}")
print(f"So anh trong tap Hold-out Test: {len(test_ids)}")


def lay_ground_truth(img_id):
    """Tra ve (duong_dan_anh, boxes_xyxy, labels) cua 1 anh trong tap test."""
    img_info = images_info[img_id]
    duong_dan = os.path.join(IMAGE_DIR, img_info['file_name'])

    boxes = []
    labels = []
    for ann in img_to_anns.get(img_id, []):
        x, y, w, h = ann['bbox']
        boxes.append([x, y, x + w, y + h])      # COCO [x,y,w,h] -> Pascal VOC [x1,y1,x2,y2]
        labels.append(int(ann['category_id']) - 1)  # JSON chay 1..7 -> ta can 0..6

    return duong_dan, np.array(boxes, dtype=np.float32).reshape(-1, 4), np.array(labels, dtype=np.int64)


# Loc bo nhung anh bi thieu file vat ly de vong lap khong bi vang loi giua chung
test_samples = []
for img_id in test_ids:
    duong_dan, boxes, labels = lay_ground_truth(img_id)
    if os.path.exists(duong_dan):
        test_samples.append((img_id, duong_dan, boxes, labels))

tong_so_bbox = sum(len(mau[2]) for mau in test_samples)
print(f"So anh thuc su doc duoc: {len(test_samples)}")
print(f"Tong so bounding box trong tap test: {tong_so_bbox}")

## Cell 5: Hàm suy luận cho nhóm `ultralytics` (YOLOv8 & RT-DETR)

YOLOv8 và RT-DETR dùng chung thư viện nên viết chung một hàm (nguyên tắc DRY). Điểm khác nhau duy nhất là tham số `imgsz` được truyền từ ngoài vào.

Thư viện `ultralytics` tự động quy đổi tọa độ đầu ra về hệ pixel của **ảnh gốc**, nên không cần scale lại thủ công.

In [ ]:
from ultralytics import YOLO, RTDETR


def nap_model_ultralytics(duong_dan_weight, la_rtdetr=False):
    """Nap model YOLOv8 hoac RT-DETR tu file .pt da train."""
    model = RTDETR(duong_dan_weight) if la_rtdetr else YOLO(duong_dan_weight)
    model.to(DEVICE)
    return model


def du_doan_ultralytics(model, duong_dan_anh, imgsz, conf):
    """Chay 1 anh, tra ve (boxes_xyxy, scores, labels) duoi dang numpy."""
    ket_qua = model.predict(
        source=duong_dan_anh,
        imgsz=imgsz,
        conf=conf,
        iou=IOU_NMS,
        max_det=MAX_DET,
        device=DEVICE,
        verbose=False,
    )[0]

    if ket_qua.boxes is None or len(ket_qua.boxes) == 0:
        return np.zeros((0, 4), np.float32), np.zeros((0,), np.float32), np.zeros((0,), np.int64)

    boxes = ket_qua.boxes.xyxy.cpu().numpy().astype(np.float32)
    scores = ket_qua.boxes.conf.cpu().numpy().astype(np.float32)
    labels = ket_qua.boxes.cls.cpu().numpy().astype(np.int64)
    return boxes, scores, labels

## Cell 6: Dựng lại và nạp mô hình Faster R-CNN

Đây là phần dễ sai nhất vì Faster R-CNN dùng PyTorch thuần, phải **dựng lại đúng y kiến trúc lúc train** rồi mới nạp được `state_dict`. Ba điểm phải khớp tuyệt đối:

1. **`num_classes = 8`** — 7 biển báo + 1 lớp nền (background) bắt buộc của PyTorch.
2. **Bộ Anchor K-Means** — mạng RPN đã học cách co giãn hộp *dựa trên* các anchor này. Nếu nạp anchor khác lúc train, tọa độ dự đoán sẽ lệch bét. Vì vậy đoạn code K-Means dưới đây được **giữ nguyên 100%** như trong `train_faster_rcnn.ipynb`, kể cả cách sắp xếp `np.sort`.
3. **`min_size=800`** — notebook train gọi `fasterrcnn_resnet50_fpn(pretrained=True)` và **không đụng vào tham số resize**, tức là dùng đúng giá trị mặc định `min_size=800, max_size=1333` của torchvision. Con số này khớp với spec mục 2.2 ("tự động scale sao cho cạnh nhỏ nhất là 800px"). Ở đây ta ghi ra tường minh cho dễ đọc chứ không phải đổi giá trị.

In [ ]:
import torchvision
from torchvision.models.detection.anchor_utils import AnchorGenerator
from sklearn.cluster import KMeans
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Chay lai K-Means y het luc train de sinh ra dung bo anchor cu.
# LUU Y: giu nguyen ca cach sort nay, KHONG duoc "sua cho dep" -
# vi doi cong thuc thi anchor se lech voi bo anchor ma RPN da hoc.
all_boxes = []
for ann in coco_data['annotations']:
    all_boxes.append([ann['bbox'][2], ann['bbox'][3]])

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
kmeans.fit(all_boxes)
centers = np.sort(kmeans.cluster_centers_, axis=0)
anchor_sizes_kmeans = tuple(int(center[0]) for center in centers)

ANCHOR_SIZES = tuple((size,) for size in anchor_sizes_kmeans)
ASPECT_RATIOS = ((1.0,),) * len(ANCHOR_SIZES)
print(f"5 kich thuoc Anchor tai lap duoc: {anchor_sizes_kmeans}")


def nap_model_faster_rcnn(duong_dan_weight):
    """Dung lai kien truc Faster R-CNN giong luc train roi nap trong so."""
    # weights=None vi ta se ghi de toan bo bang trong so da train (khoi tai ban COCO cho phi thoi gian).
    # min_size/max_size la gia tri mac dinh cua torchvision, chinh la cai luc train da dung.
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None,
        weights_backbone=None,
        num_classes=8,
        min_size=800,
        max_size=1333,
    )
    model.rpn.anchor_generator = AnchorGenerator(sizes=ANCHOR_SIZES, aspect_ratios=ASPECT_RATIOS)

    trong_so = torch.load(duong_dan_weight, map_location=DEVICE)
    model.load_state_dict(trong_so)
    model.to(DEVICE)
    model.eval()
    return model


# Tien xu ly phai giong het luc train (bo phan Random Crop vi day la luc test)
transform_test = A.Compose([A.Normalize(), ToTensorV2()])


def du_doan_faster_rcnn(model, duong_dan_anh, conf):
    """Chay 1 anh qua Faster R-CNN, tra ve (boxes_xyxy, scores, labels) numpy."""
    anh = cv2.imread(duong_dan_anh)
    anh = cv2.cvtColor(anh, cv2.COLOR_BGR2RGB)
    tensor_anh = transform_test(image=anh)['image'].to(DEVICE)

    with torch.no_grad():
        ket_qua = model([tensor_anh])[0]

    boxes = ket_qua['boxes'].cpu().numpy().astype(np.float32)
    scores = ket_qua['scores'].cpu().numpy().astype(np.float32)
    # Faster R-CNN xuat nhan 1..7 (0 la background) -> tru 1 de ve 0..6 cho dong bo
    labels = ket_qua['labels'].cpu().numpy().astype(np.int64) - 1

    # Loc theo nguong tin cay va cat ngon dung MAX_DET giong 2 model kia cho cong bang
    giu_lai = scores >= conf
    boxes, scores, labels = boxes[giu_lai], scores[giu_lai], labels[giu_lai]

    thu_tu = np.argsort(-scores)[:MAX_DET]
    return boxes[thu_tu], scores[thu_tu], labels[thu_tu]

## Cell 7: Hàm đo tốc độ suy luận (có warm-up GPU)

Lần đầu chạy, GPU phải nạp nhân CUDA và cấp phát bộ nhớ nên rất chậm — nếu bấm giờ ngay từ ảnh đầu tiên thì FPS đo được sẽ thấp giả tạo.

Vì vậy hàm dưới đây chạy nháp `WARMUP_IMAGES` ảnh trước rồi mới bấm giờ thật. Ngoài ra bắt buộc gọi `torch.cuda.synchronize()` vì lệnh GPU chạy bất đồng bộ — không đồng bộ thì đồng hồ dừng trước khi GPU tính xong, cho ra FPS cao ảo.

Một lưu ý nữa: khâu đo tốc độ chạy ở ngưỡng `conf=0.25` (ngưỡng thật của Web App) chứ không dùng ngưỡng `0.001` của phần tính mAP. Lý do là ở ngưỡng `0.001` mô hình phải xử lý hàng nghìn hộp rác qua NMS, khiến FPS đo được thấp hơn nhiều so với lúc triển khai thực tế.

In [ ]:
def do_toc_do(ham_du_doan, danh_sach_anh):
    """Do thoi gian suy luan trung binh moi anh (ms) va FPS."""
    # Giai doan 1: chay nhap cho GPU nong may, khong tinh gio
    for duong_dan in danh_sach_anh[:WARMUP_IMAGES]:
        ham_du_doan(duong_dan)

    if DEVICE.startswith('cuda'):
        torch.cuda.synchronize()

    # Giai doan 2: bam gio that
    anh_do_that = danh_sach_anh[:SPEED_IMAGES]
    bat_dau = time.perf_counter()
    for duong_dan in anh_do_that:
        ham_du_doan(duong_dan)

    if DEVICE.startswith('cuda'):
        torch.cuda.synchronize()
    ket_thuc = time.perf_counter()

    tong_thoi_gian = ket_thuc - bat_dau
    ms_moi_anh = (tong_thoi_gian / len(anh_do_that)) * 1000
    fps = len(anh_do_that) / tong_thoi_gian
    return ms_moi_anh, fps

## Cell 8: Hàm tính mAP và Ma trận nhầm lẫn

**Về mAP:** dùng `torchmetrics.MeanAveragePrecision` cho cả 3 mô hình để đảm bảo cùng một thước đo.

**Về Confusion Matrix:** thuật toán ghép cặp khá đơn giản — duyệt các hộp dự đoán theo thứ tự điểm số giảm dần, hộp nào chồng lên một hộp thật với `IoU >= 0.5` thì ghép thành một cặp. Hộp thật không ai ghép được tính là **bỏ sót (miss)**, hộp dự đoán thừa tính là **báo động giả (false alarm)**. Hai trường hợp này gom vào hàng/cột phụ tên `background`.

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision


def tinh_map(danh_sach_du_doan, danh_sach_that):
    """Tinh mAP@50, mAP@50-95 va mAP_small bang torchmetrics."""
    metric = MeanAveragePrecision(box_format='xyxy', iou_type='bbox')
    metric.update(danh_sach_du_doan, danh_sach_that)
    ket_qua = metric.compute()
    return {
        'mAP@50': float(ket_qua['map_50']),
        'mAP@50-95': float(ket_qua['map']),
        'mAP_small': float(ket_qua['map_small']),
    }


def tinh_iou_mot_nhieu(box, danh_sach_box):
    """Tinh IoU giua 1 hop voi nhieu hop khac."""
    if len(danh_sach_box) == 0:
        return np.zeros((0,), dtype=np.float32)

    x1 = np.maximum(box[0], danh_sach_box[:, 0])
    y1 = np.maximum(box[1], danh_sach_box[:, 1])
    x2 = np.minimum(box[2], danh_sach_box[:, 2])
    y2 = np.minimum(box[3], danh_sach_box[:, 3])

    giao_nhau = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    dien_tich_box = (box[2] - box[0]) * (box[3] - box[1])
    dien_tich_ds = (danh_sach_box[:, 2] - danh_sach_box[:, 0]) * (danh_sach_box[:, 3] - danh_sach_box[:, 1])

    hop_nhau = dien_tich_box + dien_tich_ds - giao_nhau
    return giao_nhau / np.maximum(hop_nhau, 1e-9)


def tinh_confusion_matrix(danh_sach_du_doan, danh_sach_that, nguong_iou=0.5):
    """Ghep cap du doan voi ground truth theo IoU, tra ve ma tran 8x8 (7 lop + background)."""
    kich_thuoc = NUM_CLASSES + 1  # chi so cuoi cung danh cho background
    ma_tran = np.zeros((kich_thuoc, kich_thuoc), dtype=np.int64)

    for du_doan, that in zip(danh_sach_du_doan, danh_sach_that):
        boxes_dd = du_doan['boxes'].numpy()
        scores_dd = du_doan['scores'].numpy()
        labels_dd = du_doan['labels'].numpy()
        boxes_that = that['boxes'].numpy()
        labels_that = that['labels'].numpy()

        # Chi giu cac du doan du tu tin, va xep diem cao truoc de uu tien ghep
        giu_lai = scores_dd >= CONF_FOR_MATRIX
        boxes_dd, labels_dd, scores_dd = boxes_dd[giu_lai], labels_dd[giu_lai], scores_dd[giu_lai]
        thu_tu = np.argsort(-scores_dd)
        boxes_dd, labels_dd = boxes_dd[thu_tu], labels_dd[thu_tu]

        da_ghep = np.zeros(len(boxes_that), dtype=bool)

        for i in range(len(boxes_dd)):
            iou = tinh_iou_mot_nhieu(boxes_dd[i], boxes_that)
            iou[da_ghep] = -1  # hop that nao ghep roi thi khong xet nua

            if len(iou) > 0 and iou.max() >= nguong_iou:
                vi_tri = int(iou.argmax())
                da_ghep[vi_tri] = True
                ma_tran[labels_that[vi_tri], labels_dd[i]] += 1
            else:
                # Du doan thua ra, khong khop vat the that nao -> bao dong gia
                ma_tran[NUM_CLASSES, labels_dd[i]] += 1

        # Nhung hop that con sot lai -> model bo sot
        for vi_tri in np.where(~da_ghep)[0]:
            ma_tran[labels_that[vi_tri], NUM_CLASSES] += 1

    return ma_tran


def ve_confusion_matrix(ma_tran, ten_model):
    """Ve ma tran nhoi lan va luu ra file PNG."""
    nhan = CLASS_NAMES + ['background']

    fig, ax = plt.subplots(figsize=(9, 7.5))
    ax.imshow(ma_tran, cmap='Blues')

    ax.set_xticks(range(len(nhan)))
    ax.set_yticks(range(len(nhan)))
    ax.set_xticklabels(nhan, rotation=45, ha='right')
    ax.set_yticklabels(nhan)
    ax.set_xlabel('Mo hinh du doan')
    ax.set_ylabel('Nhan thuc te (Ground Truth)')
    ax.set_title(f'Confusion Matrix - {ten_model} (IoU>=0.5, conf>={CONF_FOR_MATRIX})')

    # Ghi con so vao tung o cho de doc
    nguong_mau = ma_tran.max() / 2 if ma_tran.max() > 0 else 1
    for i in range(len(nhan)):
        for j in range(len(nhan)):
            ax.text(j, i, ma_tran[i, j], ha='center', va='center',
                    color='white' if ma_tran[i, j] > nguong_mau else 'black', fontsize=9)

    plt.tight_layout()
    duong_dan_luu = os.path.join(OUTPUT_DIR, f'confusion_matrix_{ten_model.replace(" ", "_")}.png')
    plt.savefig(duong_dan_luu, dpi=130)
    plt.show()
    print(f"Da luu: {duong_dan_luu}")

## Cell 9: Vòng lặp đánh giá dùng chung cho cả 3 mô hình

Tách riêng thành một hàm để không phải copy-paste 3 lần (nguyên tắc DRY). Hàm nhận vào một hàm dự đoán bất kỳ, chạy hết tập test rồi trả về đủ mAP, tốc độ và ma trận nhầm lẫn.

In [ ]:
from tqdm import tqdm


def danh_gia_mot_model(ten_model, ham_du_doan, ham_du_doan_toc_do, imgsz_ghi_chu):
    """Chay het tap test voi 1 model, tra ve dict ket qua day du.

    ham_du_doan     : chay o nguong conf rat thap, dung de tinh mAP cho chuan.
    ham_du_doan_toc_do: chay o nguong conf thuc te (0.25) giong luc trien khai Web App,
                        dung rieng cho khau do FPS de con so phan anh dung tinh huong su dung.
    """
    danh_sach_du_doan = []
    danh_sach_that = []

    for img_id, duong_dan, boxes_that, labels_that in tqdm(test_samples, desc=f"Danh gia {ten_model}"):
        try:
            boxes_dd, scores_dd, labels_dd = ham_du_doan(duong_dan)
        except Exception as loi:
            # Mot anh loi thi bo qua, khong duoc lam sap ca vong lap
            print(f"Bo qua anh {duong_dan}: {loi}")
            continue

        danh_sach_du_doan.append({
            'boxes': torch.tensor(boxes_dd, dtype=torch.float32).reshape(-1, 4),
            'scores': torch.tensor(scores_dd, dtype=torch.float32).reshape(-1),
            'labels': torch.tensor(labels_dd, dtype=torch.int64).reshape(-1),
        })
        danh_sach_that.append({
            'boxes': torch.tensor(boxes_that, dtype=torch.float32).reshape(-1, 4),
            'labels': torch.tensor(labels_that, dtype=torch.int64).reshape(-1),
        })

    print(f"Dang tinh mAP cho {ten_model}...")
    ket_qua_map = tinh_map(danh_sach_du_doan, danh_sach_that)

    print(f"Dang do toc do cho {ten_model} (warm-up {WARMUP_IMAGES} anh)...")
    danh_sach_anh = [mau[1] for mau in test_samples]
    ms_moi_anh, fps = do_toc_do(ham_du_doan_toc_do, danh_sach_anh)

    ma_tran = tinh_confusion_matrix(danh_sach_du_doan, danh_sach_that)

    return {
        'Model': ten_model,
        'imgsz': imgsz_ghi_chu,
        'mAP@50': ket_qua_map['mAP@50'],
        'mAP@50-95': ket_qua_map['mAP@50-95'],
        'mAP_small': ket_qua_map['mAP_small'],
        'Inference (ms/anh)': ms_moi_anh,
        'FPS': fps,
        '_confusion_matrix': ma_tran,
    }

## Cell 10: Chạy đánh giá lần lượt 3 mô hình

Nạp xong mỗi mô hình thì giải phóng VRAM ngay bằng `del` + `empty_cache()`, tránh mô hình sau bị thiếu bộ nhớ (Faster R-CNN và RT-DETR đều khá nặng).

In [ ]:
ket_qua_tong = []

# ---- Model 1: YOLOv8s-P2 (imgsz=1280 theo spec muc 1.2) ----
model_yolo = nap_model_ultralytics(YOLO_WEIGHT, la_rtdetr=False)
ket_qua_tong.append(danh_gia_mot_model(
    'YOLOv8s-P2',
    lambda duong_dan: du_doan_ultralytics(model_yolo, duong_dan, 1280, CONF_FOR_MAP),
    lambda duong_dan: du_doan_ultralytics(model_yolo, duong_dan, 1280, CONF_FOR_MATRIX),
    imgsz_ghi_chu='1280',
))
del model_yolo
torch.cuda.empty_cache()

# ---- Model 2: Faster R-CNN (torchvision tu scale canh nho nhat ve 800) ----
model_frcnn = nap_model_faster_rcnn(FRCNN_WEIGHT)
ket_qua_tong.append(danh_gia_mot_model(
    'Faster R-CNN R50-FPN',
    lambda duong_dan: du_doan_faster_rcnn(model_frcnn, duong_dan, CONF_FOR_MAP),
    lambda duong_dan: du_doan_faster_rcnn(model_frcnn, duong_dan, CONF_FOR_MATRIX),
    imgsz_ghi_chu='800 (tu scale)',
))
del model_frcnn
torch.cuda.empty_cache()

# ---- Model 3: RT-DETR-L (imgsz=640 theo spec muc 3.2, da ha tu 1280 de chong OOM) ----
model_rtdetr = nap_model_ultralytics(RTDETR_WEIGHT, la_rtdetr=True)
ket_qua_tong.append(danh_gia_mot_model(
    'RT-DETR-L',
    lambda duong_dan: du_doan_ultralytics(model_rtdetr, duong_dan, 640, CONF_FOR_MAP),
    lambda duong_dan: du_doan_ultralytics(model_rtdetr, duong_dan, 640, CONF_FOR_MATRIX),
    imgsz_ghi_chu='640',
))
del model_rtdetr
torch.cuda.empty_cache()

print("\nDa danh gia xong ca 3 mo hinh!")

## Cell 11: Vẽ 3 Ma trận nhầm lẫn

Đọc ma trận theo hàng: hàng `Warning signs` cột `background` cho biết model **bỏ sót** bao nhiêu biển cảnh báo. Ô nằm ngoài đường chéo chính là chỗ model **nhầm lẫn giữa hai loại biển** — đây chính là điểm mù cần phân tích trong báo cáo.

In [ ]:
for ket_qua in ket_qua_tong:
    ve_confusion_matrix(ket_qua['_confusion_matrix'], ket_qua['Model'])

## Cell 12: Bảng tổng hợp cuối cùng

Đây là bảng số liệu để dán thẳng vào báo cáo môn học.

In [ ]:
# Bo cot ma tran ra khoi DataFrame vi no la mang numpy, khong hien thi dep trong bang
bang_ket_qua = pd.DataFrame([
    {khoa: gia_tri for khoa, gia_tri in ket_qua.items() if khoa != '_confusion_matrix'}
    for ket_qua in ket_qua_tong
])

# Lam tron cho de doc
for cot in ['mAP@50', 'mAP@50-95', 'mAP_small']:
    bang_ket_qua[cot] = (bang_ket_qua[cot] * 100).round(2)
bang_ket_qua['Inference (ms/anh)'] = bang_ket_qua['Inference (ms/anh)'].round(2)
bang_ket_qua['FPS'] = bang_ket_qua['FPS'].round(2)

bang_ket_qua = bang_ket_qua.rename(columns={
    'mAP@50': 'mAP@50 (%)',
    'mAP@50-95': 'mAP@50-95 (%)',
    'mAP_small': 'mAP_small (%)',
})

duong_dan_csv = os.path.join(OUTPUT_DIR, 'final_comparison_table.csv')
bang_ket_qua.to_csv(duong_dan_csv, index=False)

print(f"So anh trong tap Hold-out Test: {len(test_samples)}")
print(f"Thiet bi do toc do: {torch.cuda.get_device_name(0) if DEVICE.startswith('cuda') else 'CPU'}")
print(f"Da luu bang ket qua: {duong_dan_csv}\n")

bang_ket_qua

## Cell 13: Ghi chú bắt buộc cho báo cáo

Chạy cell này để in ra phần chú thích. **Bắt buộc dán kèm bảng số liệu** để bảo vệ tính trung thực của thí nghiệm trước hội đồng.

In [ ]:
ghi_chu = f"""
GHI CHU VE DIEU KIEN THI NGHIEM
--------------------------------
1. Tap Test: {len(test_samples)} anh, tai lap tu 20% du lieu co nhan bang random.seed(42),
   dung phep chia ma YOLOv8 va RT-DETR da su dung luc huan luyen.
2. Thu muc traffic_public_test/ KHONG duoc dung de cham diem vi khong co nhan.
   Ca 3 mo hinh deu chi doc du lieu tu traffic_train/, KHONG co Data Leakage
   tu tap test cua cuoc thi.
3. LUU Y QUAN TRONG: Faster R-CNN luc train dung random_split() khong set seed
   va cat 90/10, nen khoang 90% anh trong tap test nay no da hoc qua roi.
   => Diem cua Faster R-CNN pha lan phan "hoc thuoc long" (Overfitting),
      trong khi YOLOv8 va RT-DETR bi cham tren anh hoan toan moi.
      Day la chenh lech dieu kien danh gia, khong phai gian lan du lieu.
4. mAP cua ca 3 model deu tinh bang torchmetrics (cung mot cong thuc) de dam bao cong bang.
5. Toc do do o che do batch=1 tren {torch.cuda.get_device_name(0) if DEVICE.startswith('cuda') else 'CPU'},
   da chay warm-up {WARMUP_IMAGES} anh truoc khi bam gio tren {SPEED_IMAGES} anh.
6. Cau hinh suy luan chung: iou={IOU_NMS}, max_det={MAX_DET}.
   Nguong tin cay: conf={CONF_FOR_MAP} khi tinh mAP (de duong cong PR du diem),
   conf={CONF_FOR_MATRIX} khi do FPS va ve Confusion Matrix (dung nguong thuc te cua Web App).
"""
print(ghi_chu)